In [21]:
import pandas as pd 
import numpy as np 
df = pd.read_csv('Bangladesh_database_Final_Merged_30features.csv')
print(f"loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns ")
df.head()

loaded dataset: 4608 rows, 29 columns 


,FPAR,Avg_Salinity_Index,EVI,Rain_Temp_Ratio,Rainfall,Wind_Mean,Temp_Min,Wind_Max,Temp_Max,District,...,Avg Humidity,Heat_Stress_Days,Silt,Growth,CN_Ratio,Crop Name,Organic_Carbon,Transplant,Season,NDVI_Season_Mean
0,6.314995,560.494023,0.311631,11.23,220.43,1.452917,8.35,4.33,30.58,Bagerhat,...,55.0,0,49.52,Jan to March,8.73,Wheat,33.97,Dec,Rabi,0.511809
1,6.314995,560.494023,0.311631,11.23,220.43,1.452917,8.35,4.33,30.58,Bagerhat,...,60.0,0,49.52,Jan to March,8.73,Maize 2,33.97,Dec,Rabi,0.511809
2,6.314995,560.494023,0.311631,11.23,220.43,1.452917,8.35,4.33,30.58,Bagerhat,...,72.5,0,49.52,Dec to March,8.73,Boro,33.97,Nov,Rabi,0.511809
3,6.314995,560.494023,0.311631,11.23,220.43,1.452917,8.35,4.33,30.58,Bagerhat,...,80.0,0,49.52,Dec to March,8.73,Sweet Potato,33.97,Nov,Rabi,0.511809
4,4.169442,883.021928,0.308177,22.73,656.33,2.481885,16.01,5.86,41.26,Bagerhat,...,70.0,54,49.52,No need to do,8.73,Mango,33.97,April,Kharif 1,0.267147


In [22]:
df['AP Ratio'] = df['AP Ratio'].replace('#DIV/0!', np.nan)
df = df.dropna(subset=['AP Ratio'])
df['AP Ratio'] = pd.to_numeric(df['AP Ratio'])
df = df.dropna(subset=['AP Ratio'])
print("Kích thước dữ liệu sau khi sửa lỗi AP Ratio:", df.shape)


Kích thước dữ liệu sau khi sửa lỗi AP Ratio: (4192, 29)


In [23]:
chuoi_cols = df.select_dtypes(include=['object', 'str']).columns

for col in chuoi_cols:
    df[col] = df[col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True).str.title()

In [24]:
if 'Dominant_Soil_Texture' in df.columns:
    df['Dominant_Soil_Texture'] = df['Dominant_Soil_Texture'].str.replace(r'\s*\(.*\)', '', regex=True)
     # Sử dụng biểu thức chính quy Regex để xóa bỏ phần giải nghĩa tiếng Việt trong dấu ngoặc đơn

print("Các giá trị kết cấu đất sau khi dọn dẹp nhiễu chữ:")
print(df['Dominant_Soil_Texture'].unique())

Các giá trị kết cấu đất sau khi dọn dẹp nhiễu chữ:
<StringArray>
['Loamy', 'Clayey']
Length: 2, dtype: str


In [ ]:
# 1. Ánh xạ đồng bộ các tên tháng viết tắt thành tên tháng viết đầy đủ
thang_map = {'Jan': 'January', 'Feb': 'February', 'Mar': 'March', 'Apr': 'April', 'Aug': 'August', 'Sep': 'September', 'Oct': 'October', 'Nov': 'November', 'Dec': 'December'}
df['Transplant'] = df['Transplant'].replace(thang_map)

# 2. Thay thế giá trị rác 'No Need To Do' ở cột lịch sinh trưởng thành tháng chuẩn của mùa vụ
df['Growth'] = df['Growth'].replace('No Need To Do', 'December To February')


Hoàn thành dọn dẹp và sửa lỗi lịch thời vụ nông nghiệp.


In [26]:
# 1. Hạ tỷ lệ chỉ số hấp thụ bức xạ FPAR về đúng dải thực tế sinh học (từ 0.0 đến 1.0)
if 'FPAR' in df.columns:
    df['FPAR'] = df['FPAR'] * 0.1

# 2. Quy đổi nhiệt độ bề mặt đất từ thang đo Kelvin sang độ Celsius và xóa cột cũ đi
if 'LST_Kelvin' in df.columns:
    df['LST_C'] = df['LST_Kelvin'] - 273.15
    df = df.drop(columns=['LST_Kelvin'])

print("Mẫu dữ liệu FPAR và Nhiệt độ bề mặt LST_C sau khi quy đổi đơn vị:")
df[['FPAR', 'LST_C']].head()

Mẫu dữ liệu FPAR và Nhiệt độ bề mặt LST_C sau khi quy đổi đơn vị:


,FPAR,LST_C
0,0.631499,24.144525
1,0.631499,24.144525
2,0.631499,24.144525
3,0.631499,24.144525
4,0.416944,28.431973


In [ ]:
df.to_csv('Agri_Data_30features_Cleaned.csv', index=False, encoding='utf-8-sig')

print(f"Dữ liệu sạch : {df.shape}")

  dữ liệu sạch : (4192, 29)
